In [20]:

import boto3
 
# ── Find exactly where your files are ────────────────────────────────────────
s3 = boto3.client("s3")
bucket = "sagemaker-us-east-1-381491860224"
 
print(f"Listing all objects under: ads-508-prognostica-project/\n")
 
response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="ads-508-prognostica-project/"
)
 
if "Contents" not in response:
    print("❌ No objects found. Check the bucket name or IAM permissions.")
else:
    for obj in response["Contents"]:
        print(obj["Key"])

Listing all objects under: ads-508-prognostica-project/

❌ No objects found. Check the bucket name or IAM permissions.


In [ ]:
# Lung Cancer Training

In [17]:
import boto3
import sagemaker
import pandas as pd
import os
from sagemaker.inputs import TrainingInput
from sagemaker.xgboost.estimator import XGBoost
from sagemaker import get_execution_role
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, confusion_matrix
)

In [13]:
# Set up

In [18]:
session = sagemaker.Session()
role    = get_execution_role()
bucket  = session.default_bucket()
prefix  = "ads-508-prognostica-project/lung-cancer"
S3_FEATURES = f"s3://sagemaker-us-east-1-381491860224/{prefix}/features/"
 

In [15]:
# 2. LOAD YOUR ALREADY-SPLIT DATA FROM S3
#    Adjust S3_FEATURES to match your actual S3 path.
#    These must be CSVs with NO header and target (lung_cancer) FIRST.

In [19]:
s3_client = boto3.client("s3")
 
def verify_s3_uri(uri):
    """Raise a clear error if an S3 file doesn't exist."""
    path   = uri.replace("s3://", "")
    bkt, key = path.split("/", 1)
    try:
        s3_client.head_object(Bucket=bkt, Key=key)
        print(f"✅ Found: {uri}")
    except Exception:
        raise FileNotFoundError(
            f"\n❌ NOT FOUND: {uri}"
            f"\n   → Open data_preparation_handoff.ipynb and print S3_FEATURES"
            f"\n   → Then set S3_FEATURES manually at the top of this script."
        )
 
TRAIN_URI = S3_FEATURES + "lung_train.csv"
VAL_URI   = S3_FEATURES + "lung_val.csv"
TEST_URI  = S3_FEATURES + "lung_test.csv"
 
print("Verifying S3 paths...")
verify_s3_uri(TRAIN_URI)
verify_s3_uri(VAL_URI)
verify_s3_uri(TEST_URI)

Verifying S3 paths...


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in verify_s3_uri:8                                                                               │
│                                                                                                  │
│    5 │   path   = uri.replace("s3://", "")                                                       │
│    6 │   bkt, key = path.split("/", 1)                                                           │
│    7 │   try:                                                                                    │
│ ❱  8 │   │   s3_client.head_object(Bucket=bkt, Key=key)                                          │
│    9 │   │   print(f"✅ Found: {uri}")                                                           │
│   10 │   except Exception:                                                                       │
│   11 │   │   raise FileNotFoundError(                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                                                                     │
│    571 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    572                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1023 in _make_api_call                │
│                                                                                                  │
│   1020 │   │   │   │   "Code"                                                                    │
│   1021 │   │   │   )                                                                             │
│   1022 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1023 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1024 │   │   else:                                                                             │
│   1025 │   │   │   return parsed_response                                                        │
│   1026                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ClientError: An error occurred (404) when calling the HeadObject operation: Not Found

During handling of the above exception, another exception occurred:

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:22                                                                                   │
│                                                                                                  │
│   19 TEST_URI  = S3_FEATURES + "lung_test.csv"                                                   │
│   20                                                                                             │
│   21 print("Verifying S3 paths...")                                                              │
│ ❱ 22 verify_s3_uri(TRAIN_URI)                                                                    │
│   23 

In [5]:
S3_FEATURES = f"s3://{bucket}/{prefix}/features/"   # ← update if needed

In [6]:
# ── 2a. Re-order columns so lung_cancer is first ──────────────────────────────
#    SageMaker built-in XGBoost requires: target col FIRST, no header row.

In [7]:
def prepare_and_upload(local_path, s3_key, s3_client, bucket, target_col="lung_cancer"):
    """Load CSV, move target to col 0, strip header, upload to S3."""
    df = pd.read_csv(local_path)
 
    # Encode YES/NO → 1/0 if not already numeric
    if df[target_col].dtype == object:
        df[target_col] = df[target_col].map({"YES": 1, "NO": 0})
 
    # Target must be first column
    cols = [target_col] + [c for c in df.columns if c != target_col]
    df   = df[cols]
 
    out = local_path.replace(".csv", "_ready.csv")
    df.to_csv(out, index=False, header=False)   # No header for SageMaker
 
    s3_client.upload_file(out, bucket, s3_key)
    print(f"Uploaded → s3://{bucket}/{s3_key}  ({len(df)} rows)")
    return df  # return for local evaluation later
 
 
s3_client = boto3.client("s3")

In [8]:
# ── Download files from S3 first (they were saved in your notebook) ───────────


In [9]:
import os, boto3
 
def download_from_s3(s3_uri, local_path):
    """Download an S3 object given a full s3:// URI."""
    parts  = s3_uri.replace("s3://", "").split("/", 1)
    bkt, key = parts[0], parts[1]
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    boto3.client("s3").download_file(bkt, key, local_path)
    print(f"Downloaded {s3_uri} → {local_path}")

In [10]:
# ── Update these URIs to match what cell [43] saved ───────────────────────────


In [11]:
TRAIN_S3_URI = S3_FEATURES + "lung_train.csv"
VAL_S3_URI   = S3_FEATURES + "lung_val.csv"
TEST_S3_URI  = S3_FEATURES + "lung_test.csv"
 
download_from_s3(TRAIN_S3_URI, "data/lung_train.csv")
download_from_s3(VAL_S3_URI,   "data/lung_val.csv")
download_from_s3(TEST_S3_URI,  "data/lung_test.csv")
 
# ── Prepare (reorder + encode) and re-upload ──────────────────────────────────
train_df = prepare_and_upload("data/lung_train.csv", f"{prefix}/train/lung_train.csv", s3_client, bucket)
val_df   = prepare_and_upload("data/lung_val.csv",   f"{prefix}/val/lung_val.csv",     s3_client, bucket)
test_df  = prepare_and_upload("data/lung_test.csv",  f"{prefix}/test/lung_test.csv",   s3_client, bucket)
 
train_input = TrainingInput(f"s3://{bucket}/{prefix}/train/", content_type="text/csv")
val_input   = TrainingInput(f"s3://{bucket}/{prefix}/val/",   content_type="text/csv")


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:5                                                                                    │
│                                                                                                  │
│    2 VAL_S3_URI   = S3_FEATURES + "lung_val.csv"                                                 │
│    3 TEST_S3_URI  = S3_FEATURES + "lung_test.csv"                                                │
│    4                                                                                             │
│ ❱  5 download_from_s3(TRAIN_S3_URI, "data/lung_train.csv")                                       │
│    6 download_from_s3(VAL_S3_URI,   "data/lung_val.csv")                                         │
│    7 download_from_s3(TEST_S3_URI,  "data/lung_test.csv")                                        │
│    8                                                                                             │
│                                                                                                  │
│ in download_from_s3:8                                                                            │
│                                                                                                  │
│    5 │   parts  = s3_uri.replace("s3://", "").split("/", 1)                                      │
│    6 │   bkt, key = parts[0], parts[1]                                                           │
│    7 │   os.makedirs(os.path.dirname(local_path), exist_ok=True)                                 │
│ ❱  8 │   boto3.client("s3").download_file(bkt, key, local_path)                                  │
│    9 │   print(f"Downloaded {s3_uri} → {local_path}")                                            │
│   10                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/boto3/s3/inject.py:192 in download_file                  │
│                                                                                                  │
│   189 │   │   transfer.                                                                          │
│   190 │   """                                                                                    │
│   191 │   with S3Transfer(self, Config) as transfer:                                             │
│ ❱ 192 │   │   return transfer.download_file(                                                     │
│   193 │   │   │   bucket=Bucket,                                                                 │
│   194 │   │   │   key=Key,                                                                       │
│   195 │   │   │   filename=Filename,                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/boto3/s3/transfer.py:406 in download_file                │
│                                                                                                  │
│   403 │   │   │   bucket, key, filename, extra_args, subscribers                                 │
│   404 │   │   )                                                                                  │
│   405 │   │   try:                                                                               │
│ ❱ 406 │   │   │   future.result()                                                                │
│   407 │   │   # This is for backwards compatibility where when retries are                       │
│   408 │   │   # exceeded we need to throw the same error from boto3 instead of                   │
│   409 │   │   # s3transfer's built in RetriesExceededError as current users are                  │
│                                                            

In [ ]:
# ─────────────────────────────────────────────
# 3. HANDLE CLASS IMBALANCE
#    Your data is ~87% YES / 13% NO.
#    scale_pos_weight = count(NO) / count(YES) down-weights the majority.
# ─────────────────────────────────────────────

In [ ]:
yes_count = (train_df.iloc[:, 0] == 1).sum()
no_count  = (train_df.iloc[:, 0] == 0).sum()
scale_pos_weight = round(no_count / yes_count, 4)
print(f"scale_pos_weight = {scale_pos_weight}  (NO:{no_count} / YES:{yes_count})")

In [ ]:
# ─────────────────────────────────────────────
# 4. DEFINE XGBOOST ESTIMATOR
# ─────────────────────────────────────────────

In [ ]:
xgb = XGBoost(
    entry_point=None,               # Built-in container — no custom script needed
    framework_version="1.7-1",      # Fall back to "1.5-1" if region doesn't have it
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session,
    hyperparameters={
        # ── Task ──────────────────────────────────────────────────────────────
        "objective":          "binary:logistic",
        "eval_metric":        "auc",          # Good for imbalanced data
 
        # ── Imbalance correction ──────────────────────────────────────────────
        "scale_pos_weight":   scale_pos_weight,
 
        # ── Regularisation & depth ────────────────────────────────────────────
        "num_round":          200,
        "max_depth":          5,
        "min_child_weight":   5,              # Helps with rare minority class
        "eta":                0.1,
        "gamma":              1,
 
        # ── Sampling ─────────────────────────────────────────────────────────
        "subsample":          0.8,
        "colsample_bytree":   0.8,
 
        # ── Early stopping (stops if val AUC doesn't improve) ─────────────────
        "early_stopping_rounds": 20,
    },
)

In [ ]:
# ─────────────────────────────────────────────
# 5. TRAIN
# ─────────────────────────────────────────────

In [ ]:
xgb.fit(
    inputs={"train": train_input, "validation": val_input},
    logs=True,
    wait=True,
)
 
print("\n Training complete!")
print(f"Model artifact → {xgb.model_data}")
 